# ASAB — 국내주식 ML 예측 모델 (Colab)

구글드라이브의 수집 DB(`market_latest.db`)를 읽어 피처를 만들고, LightGBM을
워크포워드(OOS)로 검증한다. 무료 런타임(CPU)으로 충분하다.

**핵심 원칙**: 미래정보 누출 금지(피처는 t시점까지), 워크포워드로 과적합 적발,
ML이 모멘텀/시장보다 나은지 항상 비교.

In [ ]:
# 1) 구글드라이브 마운트
from google.colab import drive
drive.mount('/content/drive')

In [ ]:
# 2) 코드(레포) 가져오기 + 의존성
!git clone -q https://github.com/AhnJung-min/ASAB.git
%cd ASAB
!pip install -q lightgbm

In [ ]:
# 3) 드라이브 백업 DB 로드 → 피처 데이터셋 생성
import sys; sys.path.insert(0, '.')
from src.data.store import DataStore
from src.features import build_dataset, FEATURES

# '내 드라이브' = Colab에서 MyDrive
DB = '/content/drive/MyDrive/ASAB_backup/market_latest.db'
store = DataStore(DB)
ds = build_dataset(store, hold_days=20, max_pool=300)
store.close()
print(f'데이터셋 {len(ds):,}행, 피처 {len(FEATURES)}개')
print('피처:', FEATURES)

In [ ]:
# 4) LightGBM 워크포워드 검증 (ML vs 모멘텀 vs 시장)
from src.ml import run_ml_wf
res = run_ml_wf(ds, train_periods=36, test_periods=6, top_n=10)
print(f"OOS {res['n_test']}기간 {res['span']}")
for k, label in (('ml','ML'),('mom','모멘텀'),('mkt','시장')):
    m = res[k]
    print(f"{label:8} 누적 {m['total']*100:7.1f}%  CAGR {m['cagr']*100:6.1f}%  "
          f"샤프 {m['sharpe']:.2f}  MDD {m['mdd']*100:.1f}%")

In [ ]:
# 5) 직접 모델 학습 + 피처 중요도 (자유 실험)
import numpy as np, lightgbm as lgb
X = np.array([[r[f] for f in FEATURES] for r in ds])
y = np.array([r['fwd_ret'] for r in ds])
model = lgb.LGBMRegressor(n_estimators=300, learning_rate=0.03, num_leaves=15,
                          min_child_samples=50, subsample=0.8, colsample_bytree=0.8,
                          reg_lambda=1.0)
model.fit(X, y)
imp = sorted(zip(FEATURES, model.feature_importances_), key=lambda t: -t[1])
for name, v in imp:
    print(f'{name:16} {v}')

## 여기서부터 개량 아이디어
- **피처 추가**: `src/features.py`의 `compute_features`에 RSI 다양화, 분석데이터
  (공매도비중·외국인순매수) 등 추가 → 직교 신호로 알파 시도
- **라벨 변경**: 절대수익 대신 *시장초과수익*(fwd_ret - 시장평균)으로 학습하면
  베타가 제거되어 알파에 집중
- **분류 문제화**: '다음 기간 상위 20%인가?'를 예측(LGBMClassifier)
- **하이퍼파라미터**: num_leaves, learning_rate 등 — 단, 워크포워드 성능으로만 판단(과적합 주의)

검증은 항상 `run_ml_wf`(OOS)로. 인샘플이 좋아지는 건 의미 없음.